# 04 Baseline tanpa model: jawaban acak dan echo pertanyaan
Jawaban acak: prediksi = jawaban referensi contoh lain di set yang sama (isi berbeda, seed 42).
Echo: prediksi = teks pertanyaan. Dihitung untuk CodeQA Python test, Java test, ClassEvalQA, dan CS1QA, lalu dinilai dengan BERTScore.

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## make_baselines.py
Menulis runs/baselines/<set>_{rand,echo}.jsonl.

In [ ]:
ARGS = []
sys.argv = ['make_baselines.py']

**Sumber:** code/make_baselines.py

```text
BERTScore baselines per test set: (a) random answer from another example, (b) echo the question.
```

In [ ]:
import json, random
from pathlib import Path

In [ ]:
ROOT = Path.home() / "inahuy/kacuy"
OUT = ROOT / "codeqa_exp/runs/baselines"
SETS = {
    "pythontest": ROOT / "kc-codeqa-reproduce/data/python/test",
    "javatest": ROOT / "kc-codeqa-reproduce/data/java/test",
    "classevalqa": ROOT / "ext_data/classevalqa",
    "cs1qa": ROOT / "ext_data/cs1qa",
}

OUT.mkdir(parents=True, exist_ok=True)
for name, d in SETS.items():
    A = [x.strip() for x in (d / "test.answer").read_text(encoding="utf-8").splitlines()]
    Q = [x.strip() for x in (d / "test.question").read_text(encoding="utf-8").splitlines()]
    assert len(A) == len(Q), name
    rng = random.Random(42)
    # random other example with a different answer string (independent draw per row; duplicates like "yes" make a strict derangement impossible)
    pick = []
    for i in range(len(A)):
        j = rng.randrange(len(A))
        while A[j] == A[i]:
            j = rng.randrange(len(A))
        pick.append(j)
    with (OUT / f"{name}_rand.jsonl").open("w", encoding="utf-8") as f:
        for i in range(len(A)):
            f.write(json.dumps({"id": i, "prediction": A[pick[i]], "reference": A[i]}, ensure_ascii=False) + "\n")
    with (OUT / f"{name}_echo.jsonl").open("w", encoding="utf-8") as f:
        for i in range(len(A)):
            f.write(json.dumps({"id": i, "prediction": Q[i], "reference": A[i]}, ensure_ascii=False) + "\n")
    print("WROTE", name, len(A), flush=True)

## bertscore_xdomain.py
BERTScore semua baseline (sama dengan run_baselines.sh).

In [ ]:
VARIANTS = {
    'baselines': ['--set', 'bl_pythontest_rand=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/pythontest_rand.jsonl', '--set', 'bl_pythontest_echo=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/pythontest_echo.jsonl', '--set', 'bl_javatest_rand=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/javatest_rand.jsonl', '--set', 'bl_javatest_echo=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/javatest_echo.jsonl', '--set', 'bl_classevalqa_rand=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/classevalqa_rand.jsonl', '--set', 'bl_classevalqa_echo=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/classevalqa_echo.jsonl', '--set', 'bl_cs1qa_rand=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/cs1qa_rand.jsonl', '--set', 'bl_cs1qa_echo=/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/cs1qa_echo.jsonl', '--out', '/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp/runs/baselines/bertscore_baselines.json'],
}
RUN = 'baselines'  # ganti ke salah satu kunci VARIANTS
ARGS = VARIANTS[RUN]
sys.argv = ['bertscore_xdomain.py', *ARGS]
print(RUN, ARGS)

**Sumber:** code/bertscore_xdomain.py

In [ ]:
import argparse, json
from pathlib import Path
import bert_score

In [ ]:
E = "/home/nafisnaufal1426/inahuy/kacuy/codeqa_exp"
p = argparse.ArgumentParser()
p.add_argument("--set", action="append", required=True, help="name=path to test_predictions.jsonl")
p.add_argument("--out", required=True)
args = p.parse_args(ARGS)
out = {}
for item in args.set:
    name, path = item.split("=", 1)
    rows = [json.loads(l) for l in open(path, encoding="utf-8")]
    P, R, F = bert_score.score([r["prediction"] for r in rows], [r["reference"] for r in rows],
                               lang="en", batch_size=128, verbose=False, device="cuda")
    out[name] = {"path": path, "n": len(rows), "BERTScore_P": 100 * P.mean().item(), "BERTScore_R": 100 * R.mean().item(), "BERTScore_F1": 100 * F.mean().item()}
    print("DONE", name, json.dumps(out[name]), flush=True)
Path(args.out).write_text(json.dumps(out, indent=2), encoding="utf-8")
print("WROTE", args.out, flush=True)